<p style="font-size: 20px; color:#00ff00; text-align: center">
     Importando as bibliotecas
</p>


In [70]:
import pandas as pd

<p style="font-size: 20px; color:#00ff00; text-align: center">
   Importando e Tratando dados
</p>

In [71]:
def read_database(caminho_arquivo):
    excel_file = pd.ExcelFile(caminho_arquivo)
    planilhas = [nome for nome in excel_file.sheet_names if nome != "Parâmetros"]
    
    dfs = {}
    receita = 0
    for planilha in planilhas: 
        df = pd.read_excel(
            excel_file,
            sheet_name = planilha,
            skiprows=1,
            usecols="A:J",
            header=None
        )
        df = df.drop(columns=[7,8])
        df.columns= df.iloc[0]
        df = df[1:].reset_index(drop=True)

        if not df.empty:
            try:
                valor = df.iloc[0, 7] 
                valor_float = float(str(valor).replace(',', '.'))
                receita += valor_float
            except (ValueError, TypeError):
                ...
        chave = planilha.lower().replace(" ", "_").replace("2024", "24")
        dfs[chave] = df
        
    return dfs,receita

In [73]:
def normalizar_valor(valor):
    valor = str(valor).replace('R$', '').strip()

    if '.' in valor and ',' in valor:
        valor = valor.replace('.', '').replace(',', '.')
    elif ',' in valor:
        valor = valor.replace(',', '.')

    try:
        return float(valor)
    except ValueError:
        return 0.0

In [74]:
def limpar_dados(dfs):
    dfs_limpos = {}

    for chave, df in dfs.items():
        df = df.copy()

        
        df.columns = df.columns.astype(str).str.strip()
        colunas_para_formatar = []

        if 'Valor' in df.columns:
            df['Valor'] = df['Valor'].apply(normalizar_valor)
            df['Valor'] = df['Valor'].fillna(0)
            colunas_para_formatar.append('Valor')

        if 'Sazonalidade' in df.columns:
            df['Sazonalidade'] = pd.to_numeric(df['Sazonalidade'], errors='coerce').fillna(0).round(2)
            colunas_para_formatar.append('Sazonalidade')

        for coluna in ['Descrição', 'Data da Despesa']:
            if coluna in df.columns:
                df[coluna] = df[coluna].fillna('Desconhecido')

        if 'Data da Despesa' in df.columns:
            df['Data da Despesa'] = pd.to_datetime(df['Data da Despesa'], errors='coerce')
            df['Mes_Ano'] = df['Data da Despesa'].dt.to_period('M').astype(str)

        elif 'Vencimento' in df.columns:
            df['Vencimento'] = pd.to_datetime(df['Vencimento'], errors='coerce')
            df['Mes_Ano'] = df['Vencimento'].dt.to_period('M').astype(str)

        for col in colunas_para_formatar:
            df[col] = df[col].apply(
                lambda x: f"{x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
            )

        dfs_limpos[chave] = df

    return dfs_limpos


In [75]:
dados_limpos = limpar_dados(data)
dados_limpos

{'setembro_24': 0                                   Item Vencimento     Valor     Tipo  \
 0         Sistema de Geração de Energia  2024-09-01    260,00  Despesa   
 1                                  Cemig 2024-09-02     74,85  Despesa   
 2                     Folha de Pagamento 2024-09-05  4.392,57  Despesa   
 3                                Sistema 2024-09-05    480,10  Despesa   
 4                         Telefone fixo  2024-09-05    240,98  Despesa   
 5                                  Damae 2024-09-05     98,30  Despesa   
 6                                   FGTS 2024-09-07    375,12  Despesa   
 7                               Internet 2024-09-12    119,00  Despesa   
 8                      Movimento interno 2024-09-15  3.038,50  Receita   
 9                       Honor. Contábeis 2024-09-15    752,75  Despesa   
 10                                  Site 2024-09-16     50,00  Despesa   
 11                              Simples  2024-09-20  1.101,01  Despesa   
 12       

In [76]:
def consolidar_fluxo_de_caixa(dados):
    consolidado = []
    for chave, df in dados.items():
        if df.empty or not all(col in df.columns for col in ["Valor", "Tipo", "mes_ano"]):
            print(f"dataframe {chave} foi iginorado")
            continue

        df["Valor"] = pd.to_numeric(df["Valor"],errors="coerce").fillna(0)

        df["mes_ano"] = df["mes_ano"].astype(str)

        try:
            receita = float(df.columns[7])
        except:
            receita = 0 

        despesa_total = df[df["Tipo"].str.lower() == "despesa"]["Valor"].sum()
        lucro_bruto = receita - despesa_total

        mes_anos = df["mes_ano"].iloc[0]

        consolidado.append({
            "mes_ano": mes_anos, "receita_do_mes": receita, "despesa_total": despesa_total, "lucro_bruto": lucro_bruto 
        })
    df_consolidado = pd.DataFrame(consolidado).sort_values("mes_ano")


    df_consolidado["ds"] = pd.to_datetime(df_consolidado["mes_ano"] + "-01")
    df_consolidado = df_consolidado.sort_values("ds")
    df_consolidado["ano"] = df_consolidado["ds"].dt.year
    df_consolidado["mes"] = df_consolidado["ds"].dt.month

    df_consolidado["y"] = df_consolidado["lucro_bruto"]
    df_consolidado["variacao_mensal"] = df_consolidado["lucro_bruto"].pct_change().fillna(0).round(4)
    df_consolidado["despesas_acumulada"] = df_consolidado["despesa_total"].cumsum()
    df_consolidado["lucro_acumulado"] = df_consolidado["lucro_bruto"].cumsum()

    df_consolidado["margem_de_lucro"] = (
        (df_consolidado["lucro_bruto"] / df_consolidado["receita_do_mes"])
        .replace([float("inf"), -float("inf")], 0).round(4)
    )

    df_consolidado["alerta_risco"] = df_consolidado["lucro_bruto"].apply(
        lambda valor: "risco" if valor < 10000 else "ok"
    )

    return df_consolidado


        

<p style="font-size: 20px; color:#00ff00; text-align: center">
   Visualizando Arquivos 

</p>

In [77]:
caminho = r"C:\Users\admin\Documents\Project_Capstone\source\data\Project_Capstone.xlsx"
data,receita_total = read_database(caminho)

dados_limpos = limpar_dados(data)

df = consolidar_fluxo_de_caixa(dados_limpos)
df 

dataframe setembro_24 foi iginorado
dataframe outubro_24 foi iginorado
dataframe novembro_24 foi iginorado
dataframe dezembro_24 foi iginorado
dataframe janeiro_2025 foi iginorado
dataframe fevereiro_2025 foi iginorado


KeyError: 'mes_ano'

In [ ]:
df.to_excel(r"C:\Users\admin\Documents\Project_Capstone\source\data\dados_consolidados.xlsx",index=False)
df.to_csv(r"C:\Users\admin\Documents\Project_Capstone\source\data\dados_consolidados.csv",index=False,sep=";")